In [1]:
# required libraries
import os
import torch
import torch.nn as nn

import math

import data.processor as preprocess

import mlmodel.mlp as model_mlp
import mlmodel.softdt as model_softdt
import mlmodel.tab_transformer as model_tabtrans
import mlmodel.simple_vae as model_simple_vae

processed_data_path = os.path.join(os.getcwd(), 'data/preprocessed/')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

from attack.vae_sparsity_l1_attack import VAESparsityL1Attack
import attack.run_gridsearch as run_gridsearch

# Remove the Future warning from pytorch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
LOAD_EXIST = True

In [3]:
def pick_true_prediction(model, data):
    X_test_tensor, y_test_tensor = data

    with torch.no_grad():
        model.eval()
        test_outputs = model(X_test_tensor.to(device))
        predicted = torch.argmax(test_outputs, dim=1).float()
        prediction = (predicted == y_test_tensor.to(device))

        # Create tensors to hold true predictions' indices, inputs, and targets
        indices_of_true_predictions = []
        X_of_true_predictions = []
        y_of_true_predictions = []

        for i in range(len(prediction)):
            if prediction[i]:
                indices_of_true_predictions.append(i)
                X_of_true_predictions.append(X_test_tensor[i])
                y_of_true_predictions.append(y_test_tensor[i])

        return (
            torch.tensor(indices_of_true_predictions),
            torch.stack(X_of_true_predictions),
            torch.stack(y_of_true_predictions),
        )


In [4]:
parameter_grid_baseline = {
    '_lambda': [1], # Regularization parameter, 0.01, 0.03, 0.1, 0.3, 0.5, 0.7, 1 # 0.6, 0.7, 0.8, 0.9, 1
    'lambda_sparsity': [1], # Sparsity parameter
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.1],  # Learning rates to try, 0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value
}

In [5]:
parameter_grid_search = {
    '_lambda': [1], # Regularization parameter, # 0.6, 0.7, 0.8, 0.9, 1
    'lambda_sparsity': [0.01, 0.1, 0.25, 0.5, 0.75, 1, 1.25, 2, 5, 10], # Sparsity parameter
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.1],  # Learning rates to try
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value
}



### Adult

In [6]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/adult')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([21112, 12]), y_train: torch.Size([21112])
X_val: torch.Size([3017, 12]), y_val: torch.Size([3017])
X_test: torch.Size([6033, 12]), y_test: torch.Size([6033])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 7
The num of continues features is: 5
The num of total features is: 12
The num of binary features is: 1
Combined embedding dims: [(16, 4), (7, 3), (7, 3), (14, 4), (6, 3), (5, 3), (41, 7)]


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Load VAE model

In [7]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "adult",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-3,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/adult_0.001_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(16, 4)
    (1-2): 2 x Embedding(7, 3)
    (3): Embedding(14, 4)
    (4): Embedding(6, 3)
    (5): Embedding(5, 3)
    (6): Embedding(41, 7)
  )
  (encoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0

Load ml model

In [8]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "adult",
    "epochs": 150,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_adult.pt


In [9]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "adult",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=2, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_adult.pt


In [10]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "adult",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                      num_continuous=continues_num,
                                      dim=trans_dim,
                                      depth=3,
                                      heads=4,
                                      dim_head=16,
                                      dim_out=2,
                                      ff_dropout=0.2,
                                      attn_dropout=0.2,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_adult.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [11]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]: #"SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAESparsityL1Attack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="sparsity_l1",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.29,
        "mean_l2_distance": 0.27978459000587463,
        "mean_linf_distance": 0.14490680396556854,
        "perturbed_features_latent": 15.986206896551725,
        "perturbed_features_input": 4.551724137931035,
        "mean_confidence_successful": 0.5651445201501764,
        "mean_confidence_u

In [12]:
for model_str in ["MLP"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_search['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAESparsityL1Attack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="sparsity_l1",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_search,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/10

Testing parameters: {'_lambda': 1, 'lambda_sparsity': 0.01, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lambda_sparsity': 0.01, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 0.01,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.506,
        "mean_l2_distance": 0.6077836155891418,
        "mean_linf_distance": 0.2945929169654846,
        "perturbed_features_latent": 15.984189723320158,
        "perturbed_features_input": 4.6561264822134385,
        "mean_confidence_successful": 0.5606231379208593,
        "mean_co

### phishing_url

In [13]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/phishing_url')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([8001, 86]), y_train: torch.Size([8001])
X_val: torch.Size([1143, 86]), y_val: torch.Size([1143])
X_test: torch.Size([2286, 86]), y_test: torch.Size([2286])
The num of embedding dim for Transformer is: 2
The num of categorical features is: 1
The num of continues features is: 85
The num of total features is: 86
The num of binary features is: 28
Combined embedding dims: [(3, 2)]


In [14]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "phishing_url",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/phishing_url_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(3, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=87, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_f

Load ml model

In [15]:

mlp_config = {
    "model": "MLP",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_phishing_url.pt


In [16]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_phishing_url.pt


In [17]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "phishing_url",
    "epochs": 50,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_phishing_url.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [18]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]: # "SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAESparsityL1Attack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="phishing_url",                         # Dataset name
        model=model_str,                               # Model name
        attack="sparsity_l1",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2198
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.4,
        "mean_l2_distance": 0.8961325883865356,
        "mean_linf_distance": 0.48673996329307556,
        "perturbed_features_latent": 16.0,
        "perturbed_features_input": 59.13,
        "mean_confidence_successful": 0.7004355387529358,
        "mean_confidence_unsuccessful": 0.025653033653

### PenDigit

In [19]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/pendigit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([7693, 16]), y_train: torch.Size([7693])
X_val: torch.Size([1100, 16]), y_val: torch.Size([1100])
X_test: torch.Size([2199, 16]), y_test: torch.Size([2199])
The num of embedding dim for Transformer is: 0
The num of categorical features is: 0
The num of continues features is: 16
The num of total features is: 16
The num of binary features is: 0
Combined embedding dims: []


Load VAE model

In [20]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
    num_classes=10
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/pendigit_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList()
  (encoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=64, bias=True)
      (3): Re

Load ml model

In [21]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=10,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_pendigit.pt


In [22]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=10, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_pendigit.pt


In [23]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=[],
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=10,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')


Model loaded from models/TabTransformer_pendigit.pt


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Apply attacks - MLP + SoftDT + Tab Transfomer

In [24]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]: # "SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAESparsityL1Attack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="pendigit",                         # Dataset name
        model=model_str,                               # Model name
        attack="sparsity_l1",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2154
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lambda_sparsity': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.562,
        "mean_l2_distance": 0.9303797483444214,
        "mean_linf_distance": 0.6076347827911377,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.6274214275065293,
        "mean_confidence_unsuccessful": 0.0249694177549

In [25]:
for model_str in ["MLP"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_search['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAESparsityL1Attack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="pendigit",                         # Dataset name
        model=model_str,                               # Model name
        attack="sparsity_l1",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_search,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2154
------------------------------------------------------------
Running combination 1/10

Testing parameters: {'_lambda': 1, 'lambda_sparsity': 0.01, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Could not load existing results: Attack folder does not exist: adversarial_examples/pendigit/MLP/sparsity_l1/Try_500_inputs/_lambda_1_sparsity_0.01_lr_0.1_max_iter_300
Running attack instead for parameters: {'_lambda': 1, 'lambda_sparsity': 0.01, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}


Generating Adversarial Examples: 100%|██████████| 500/500 [02:14<00:00,  3.71it/s]


Total successful adversarial examples: 452/500
Adversarial examples confidence: Mean - 0.5371471957592293, Max - 0.9967687255702913, Min - 9.703636169433594e-05
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 0.01,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.904,
        "mean_l2_distance": 1.0918923616409302,
        "mean_linf_distance": 0.6752628684043884,
        "perturbed_features_latent": 7.997787610619469,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.5904840621963324,
        "mean_confidence_unsuccessful": 0.03489170347650846,
        "min_confidence_unsuccessful": 9.703636169433594e-05,
        "max_confidence_unsuccessful": 0.5150100588798523,
        "mean_md_distance": 2.5161418533419466,
        "outlier_rate": 0.05752212389380531,
        "outliers": "26/452"
    }
}

---------------

Generating Adversarial Examples: 100%|██████████| 500/500 [02:10<00:00,  3.83it/s]


Total successful adversarial examples: 433/500
Adversarial examples confidence: Mean - 0.5193872279208154, Max - 0.9979246314615011, Min - 8.07046890258789e-05
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 0.1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.866,
        "mean_l2_distance": 0.994843065738678,
        "mean_linf_distance": 0.6228348612785339,
        "perturbed_features_latent": 7.997690531177829,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.5955481664460436,
        "mean_confidence_unsuccessful": 0.027182953571205707,
        "min_confidence_unsuccessful": 8.07046890258789e-05,
        "max_confidence_unsuccessful": 0.3904387354850769,
        "mean_md_distance": 2.5225117542691007,
        "outlier_rate": 0.04849884526558892,
        "outliers": "21/433"
    }
}

------------------

...

Total successful adversarial examples: 173/500
Adversarial examples confidence: Mean - 0.2530408203636762, Max - 0.9998989185551181, Min - 8.58306884765625e-06
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 2,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.346,
        "mean_l2_distance": 1.010604739189148,
        "mean_linf_distance": 0.656456708908081,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.6729181670830073,
        "mean_confidence_unsuccessful": 0.030903875463234903,
        "min_confidence_unsuccessful": 8.58306884765625e-06,
        "max_confidence_unsuccessful": 0.45982682704925537,
        "mean_md_distance": 2.906166224946693,
        "outlier_rate": 0.14450867052023122,
        "outliers": "25/173"
    }
}

-----------------------------------

Generating Adversarial Examples: 100%|██████████| 500/500 [02:03<00:00,  4.06it/s]


Total successful adversarial examples: 74/500
Adversarial examples confidence: Mean - 0.13406691618612968, Max - 0.9998989185551181, Min - 6.556510925292969e-06
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 5,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.148,
        "mean_l2_distance": 1.168039321899414,
        "mean_linf_distance": 0.7550171613693237,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.7037391744387286,
        "mean_confidence_unsuccessful": 0.03510976334412893,
        "min_confidence_unsuccessful": 6.556510925292969e-06,
        "max_confidence_unsuccessful": 0.5343761742115021,
        "mean_md_distance": 3.322771192464062,
        "outlier_rate": 0.21621621621621623,
        "outliers": "16/74"
    }
}

-----------------------------------

Generating Adversarial Examples: 100%|██████████| 500/500 [02:04<00:00,  4.01it/s]

Total successful adversarial examples: 47/500
Adversarial examples confidence: Mean - 0.0944683355356101, Max - 0.9998989185551181, Min - 7.271766662597656e-06
Results: 
{
    "params": {
        "_lambda": 1,
        "lambda_sparsity": 10,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.094,
        "mean_l2_distance": 1.4520213603973389,
        "mean_linf_distance": 0.9334571361541748,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.6867947099319204,
        "mean_confidence_unsuccessful": 0.03301283973731742,
        "min_confidence_unsuccessful": 7.271766662597656e-06,
        "max_confidence_unsuccessful": 0.4853614568710327,
        "mean_md_distance": 3.9278415127537314,
        "outlier_rate": 0.425531914893617,
        "outliers": "20/47"
    }
}

Best Parameters: {'_lambda': 1, 'la